# 🏆 Capstone Project — End-to-End Image Classification Pipeline
### Introduction to Deep Learning | AgenticLabs.ng

---

## Overview

This capstone brings together everything you have learned across Modules 0–6. You will build a **complete, production-quality image classification pipeline** from scratch — from raw data to a saved, deployable model.

## 🎯 What You Will Build

1. **Data pipeline** — Download, explore, split, and preprocess a dataset
2. **Custom CNN model** — Design an appropriate architecture
3. **Complete training loop** — With regularisation, scheduling, and early stopping
4. **Evaluation** — Accuracy, confusion matrix, and per-class analysis
5. **Error analysis** — Understand where your model fails
6. **Model export** — Save and reload for inference
7. **Reflection** — Write a short analysis of your results

---

## Step 1 — Choose Your Dataset

You can use one of the following, or bring your own:

| Dataset | Classes | Samples | Difficulty |
|---|---|---|---|
| CIFAR-10 (default) | 10 | 60K | Medium |
| Fashion-MNIST | 10 | 70K | Medium |
| STL-10 | 10 | 13K (96×96) | Hard |
| Your own Kaggle dataset | Custom | — | Variable |

**For this notebook, we use CIFAR-10.** To switch datasets, change the `DATASET` variable in Step 2.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets, models
from torch.utils.data import DataLoader, Subset, random_split
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import time
import json
import os
from tqdm import tqdm

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ── Config ────────────────────────────────────────────────
DATASET     = "CIFAR10"      # Change to "FashionMNIST" or "STL10"
BATCH_SIZE  = 128
EPOCHS      = 30
LR          = 0.001
PATIENCE    = 7
SAVE_PATH   = "best_model.pth"
print(f"Dataset: {DATASET} | Batch: {BATCH_SIZE} | Epochs: {EPOCHS}")


## Step 2 — Data Pipeline

In [ ]:
# ── Data loading with augmentation ────────────────────────
CLASSES = {
    "CIFAR10":      ['airplane','automobile','bird','cat','deer',
                     'dog','frog','horse','ship','truck'],
    "FashionMNIST": ['T-shirt','Trouser','Pullover','Dress','Coat',
                     'Sandal','Shirt','Sneaker','Bag','Ankle boot'],
    "STL10":        ['airplane','bird','car','cat','deer',
                     'dog','horse','monkey','ship','truck'],
}
CLASS_NAMES = CLASSES[DATASET]

if DATASET == "CIFAR10":
    MEAN, STD = (0.4914,0.4822,0.4465), (0.2023,0.1994,0.2010)
    train_tf = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD)
    ])
    test_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    raw_train = datasets.CIFAR10('./data', train=True,  download=True, transform=train_tf)
    raw_test  = datasets.CIFAR10('./data', train=False, download=True, transform=test_tf)

elif DATASET == "FashionMNIST":
    MEAN, STD = (0.2860,), (0.3530,)
    train_tf = transforms.Compose([
        transforms.RandomHorizontalFlip(), transforms.RandomRotation(10),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    test_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    raw_train = datasets.FashionMNIST('./data', train=True,  download=True, transform=train_tf)
    raw_test  = datasets.FashionMNIST('./data', train=False, download=True, transform=test_tf)

# Train/Val split (80/20)
val_size   = int(0.2 * len(raw_train))
train_size = len(raw_train) - val_size
train_set, val_set = random_split(raw_train, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(raw_test,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_set):,} | Val: {len(val_set):,} | Test: {len(raw_test):,}")
print(f"Classes: {CLASS_NAMES}")


In [ ]:
# ── Exploratory Data Analysis ─────────────────────────────
# Class distribution
from collections import Counter
raw_noaug = datasets.CIFAR10('./data', train=True,
    transform=transforms.ToTensor()) if DATASET=="CIFAR10" else datasets.FashionMNIST(
    './data', train=True, transform=transforms.ToTensor())

labels_all = [raw_noaug[i][1] for i in range(len(raw_noaug))]
counts = Counter(labels_all)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribution bar chart
axes[0].bar([CLASS_NAMES[i] for i in range(10)], [counts[i] for i in range(10)],
             color='#4C72B0', edgecolor='white')
axes[0].set_title("Class Distribution (Training Set)")
axes[0].set_xlabel("Class"); axes[0].set_ylabel("Count")
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# Sample images per class
axes[1].axis('off')
fig2, ax_grid = plt.subplots(2, 5, figsize=(13, 5))
for cls_idx in range(10):
    indices = [i for i, l in enumerate(labels_all) if l == cls_idx][:1]
    img, _ = raw_noaug[indices[0]]
    ax = ax_grid[cls_idx//5, cls_idx%5]
    if img.shape[0] == 1:
        ax.imshow(img.squeeze(), cmap='gray')
    else:
        ax.imshow(img.permute(1,2,0))
    ax.set_title(CLASS_NAMES[cls_idx], fontsize=10); ax.axis('off')

plt.suptitle(f"{DATASET} — One Sample Per Class", fontsize=12)
plt.tight_layout(); plt.show()
print(f"Dataset is balanced: {min(counts.values())} to {max(counts.values())} samples per class")


## Step 3 — Model Architecture

In [ ]:
# ── Custom CNN Architecture ───────────────────────────────
in_channels = 1 if DATASET == "FashionMNIST" else 3

class CapstoneModel(nn.Module):
    """
    Custom CNN for image classification.
    Architecture: 4 conv blocks + classifier head
    """
    def __init__(self, in_channels=3, num_classes=10):
        super().__init__()
        
        def conv_block(in_ch, out_ch, pool=True):
            layers = [
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
            ]
            if pool:
                layers.append(nn.MaxPool2d(2))
            layers.append(nn.Dropout2d(0.1))
            return nn.Sequential(*layers)
        
        self.block1 = conv_block(in_channels, 64)   # → 16×16 or 48×48
        self.block2 = conv_block(64, 128)            # → 8×8 or 24×24
        self.block3 = conv_block(128, 256)           # → 4×4 or 12×12
        self.block4 = conv_block(256, 256, pool=False)
        
        self.global_avg_pool = nn.AdaptiveAvgPool2d((4, 4))
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256*4*4, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.global_avg_pool(x)
        return self.classifier(x)

model = CapstoneModel(in_channels=in_channels, num_classes=10).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable:,}")


## Step 4 — Complete Training Loop

In [ ]:
# ── Training with early stopping + LR scheduling ──────────
class EarlyStopping:
    def __init__(self, patience=7, delta=0.001):
        self.patience = patience; self.delta = delta
        self.best_loss = float('inf'); self.counter = 0
        self.stop = False; self.best_weights = None
    
    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss; self.counter = 0
            self.best_weights = {k:v.clone() for k,v in model.state_dict().items()}
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
    
    def restore(self, model):
        if self.best_weights:
            model.load_state_dict(self.best_weights)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimiser = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=EPOCHS)
es        = EarlyStopping(patience=PATIENCE)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lr": []}

print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>8} | {'Val Acc':>7} | {'LR':>8}")
print("─" * 65)

start = time.time()
for epoch in range(1, EPOCHS+1):
    # ── Train ──────────────────────────────────────────────
    model.train()
    tl = tc = tt = 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimiser.zero_grad()
        out = model(X); loss = criterion(out, y)
        loss.backward(); optimiser.step()
        tl += loss.item(); tc += (out.argmax(1)==y).sum().item(); tt += y.size(0)
    
    # ── Validate ────────────────────────────────────────────
    model.eval()
    vl = vc = vt = 0
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)
            out = model(X); loss = criterion(out, y)
            vl += loss.item(); vc += (out.argmax(1)==y).sum().item(); vt += y.size(0)
    
    tr_loss, tr_acc = tl/len(train_loader), tc/tt*100
    vl_loss, vl_acc = vl/len(val_loader),   vc/vt*100
    current_lr      = optimiser.param_groups[0]['lr']
    
    for k, v in zip(['train_loss','val_loss','train_acc','val_acc','lr'],
                     [tr_loss, vl_loss, tr_acc, vl_acc, current_lr]):
        history[k].append(v)
    
    scheduler.step()
    es.step(vl_loss, model)
    
    marker = " ✅" if vl_loss == es.best_loss else f" ⏳({es.counter}/{PATIENCE})"
    print(f"{epoch:5d} | {tr_loss:10.4f} | {tr_acc:8.2f}% | {vl_loss:8.4f} | {vl_acc:6.2f}%"
          f" | {current_lr:.2e}{marker}")
    
    if es.stop:
        print(f"\n🛑 Early stopping at epoch {epoch}")
        break

es.restore(model)
elapsed = time.time() - start
print(f"\nTotal training time : {elapsed:.0f}s")
print(f"Best val accuracy   : {max(history['val_acc']):.2f}%")


In [ ]:
# ── Plot full training history ────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ep = range(1, len(history['train_loss'])+1)

axes[0].plot(ep, history['train_loss'], 'b-', lw=2, label='Train')
axes[0].plot(ep, history['val_loss'],   'r-', lw=2, label='Val')
axes[0].set_title("Loss Curves"); axes[0].set_xlabel("Epoch")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep, history['train_acc'], 'b-', lw=2, label='Train')
axes[1].plot(ep, history['val_acc'],   'r-', lw=2, label='Val')
axes[1].set_title("Accuracy Curves"); axes[1].set_xlabel("Epoch")
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(ep, history['lr'], 'g-', lw=2)
axes[2].set_title("Learning Rate Schedule"); axes[2].set_xlabel("Epoch")
axes[2].set_yscale('log'); axes[2].grid(alpha=0.3)

plt.suptitle(f"{DATASET} — Full Training History", fontsize=13)
plt.tight_layout(); plt.show()


## Step 5 — Test Set Evaluation

In [ ]:
# ── Final test evaluation ─────────────────────────────────
model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        logits = model(X)
        probs  = torch.softmax(logits, dim=1)
        preds  = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

overall_acc = (all_preds == all_labels).mean() * 100
print(f"\n{'='*50}")
print(f"  Final Test Accuracy: {overall_acc:.2f}%")
print(f"{'='*50}")
print("\nPer-class results:")
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))


In [ ]:
# ── Confusion matrix ─────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title(f"Confusion Matrix — {DATASET} (Accuracy: {overall_acc:.2f}%)")
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

# Per-class accuracy
per_class_acc = cm.diagonal() / cm.sum(axis=1) * 100
plt.figure(figsize=(10, 4))
bars = plt.bar(CLASS_NAMES, per_class_acc, color=['#4C72B0' if a >= overall_acc else '#DD8452' for a in per_class_acc])
plt.axhline(overall_acc, color='red', ls='--', lw=1.5, label=f'Overall: {overall_acc:.1f}%')
plt.ylabel("Accuracy (%)"); plt.title("Per-Class Accuracy")
plt.xticks(rotation=45, ha='right'); plt.legend(); plt.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


## Step 6 — Error Analysis

In [ ]:
# ── Show worst predictions ────────────────────────────────
# Find high-confidence wrong predictions
wrong_mask = all_preds != all_labels
wrong_confidence = all_probs[wrong_mask].max(axis=1)
wrong_indices    = np.where(wrong_mask)[0]
sort_idx         = np.argsort(wrong_confidence)[::-1][:12]

raw_test_noaug = datasets.CIFAR10('./data', train=False,
    transform=transforms.ToTensor()) if DATASET=="CIFAR10" else     datasets.FashionMNIST('./data', train=False, transform=transforms.ToTensor())

fig, axes = plt.subplots(3, 4, figsize=(13, 9))
for i, ax in enumerate(axes.ravel()):
    if i >= len(sort_idx): ax.axis('off'); continue
    idx      = wrong_indices[sort_idx[i]]
    img, _   = raw_test_noaug[idx]
    true_cls = CLASS_NAMES[all_labels[idx]]
    pred_cls = CLASS_NAMES[all_preds[idx]]
    conf     = wrong_confidence[sort_idx[i]]
    
    if img.shape[0] == 1:
        ax.imshow(img.squeeze(), cmap='gray')
    else:
        ax.imshow(img.permute(1,2,0))
    ax.set_title(f"True: {true_cls}\nPred: {pred_cls} ({conf:.2f})", fontsize=9,
                  color='red' if true_cls != pred_cls else 'green')
    ax.axis('off')

plt.suptitle("Highest-Confidence Wrong Predictions (Error Analysis)", fontsize=13)
plt.tight_layout(); plt.show()


## Step 7 — Save & Export the Model

In [ ]:
# ── Save model checkpoint ─────────────────────────────────
checkpoint = {
    'model_state_dict':  model.state_dict(),
    'model_architecture': str(model),
    'class_names':       CLASS_NAMES,
    'test_accuracy':     overall_acc,
    'training_history':  history,
    'config': {
        'dataset': DATASET, 'epochs_trained': len(history['train_loss']),
        'best_val_acc': max(history['val_acc']), 'lr': LR
    }
}
torch.save(checkpoint, SAVE_PATH)
print(f"✅ Model saved to: {SAVE_PATH}")
print(f"   Checkpoint size: {os.path.getsize(SAVE_PATH)/1e6:.1f} MB")

# ── Reload and verify ─────────────────────────────────────
ckpt    = torch.load(SAVE_PATH, map_location=device)
model2  = CapstoneModel(in_channels=in_channels, num_classes=10).to(device)
model2.load_state_dict(ckpt['model_state_dict'])
model2.eval()

# Quick inference on 1 sample
img, label = raw_test_noaug[0]
with torch.no_grad():
    out  = model2(img.unsqueeze(0).to(device))
    pred = out.argmax(1).item()

print(f"\nReloaded model inference check:")
print(f"  True class  : {CLASS_NAMES[label]}")
print(f"  Predicted   : {CLASS_NAMES[pred]}")
print(f"  Confidence  : {torch.softmax(out, 1).max().item():.3f}")
print("\n✅ Model reload successful!")


## Step 8 — Project Reflection ✍️

**Complete the following reflection in the cell below. This is your written submission.**

Answer each question in 2–5 sentences:

1. **Architecture choice**: Why did you choose this number of layers and filters? What did you try that didn't work?
2. **Regularisation**: Which regularisation technique had the biggest impact on your validation accuracy?
3. **Difficult classes**: Which classes did your model confuse most often? Why might that be?
4. **What surprised you?** What result or behaviour was unexpected?
5. **Next steps**: If you had more time, what would you try to improve accuracy further?


In [ ]:
# ── 📝 Write your reflection here ────────────────────────
reflection = """
1. ARCHITECTURE CHOICE:
   [Your answer here]

2. REGULARISATION:
   [Your answer here]

3. DIFFICULT CLASSES:
   [Your answer here]

4. WHAT SURPRISED ME:
   [Your answer here]

5. NEXT STEPS:
   [Your answer here]
"""

print("Capstone Project Reflection")
print("=" * 50)
print(reflection)

# Save reflection to disk
with open("capstone_reflection.txt", "w") as f:
    f.write(reflection)
print("\n✅ Reflection saved to capstone_reflection.txt")


## 🏁 Capstone Complete — Submission Checklist

Before submitting, verify:

- [ ] Dataset was loaded and explored (class distribution plotted)
- [ ] Custom CNN model was designed and explained
- [ ] Training ran for at least 10 epochs with loss curves plotted
- [ ] Early stopping and LR scheduling were applied
- [ ] Confusion matrix and per-class accuracy are shown
- [ ] Error analysis (worst predictions) is completed
- [ ] Model was saved and reloaded successfully
- [ ] Reflection is written with all 5 questions answered

**Congratulations on completing the Introduction to Deep Learning course at AgenticLabs! 🎉**

---

### What's Next?

- 🤖 **Agentic AI Program** — Build autonomous AI agents with LLMs
- 🧪 **AI & ML Program** — Production ML systems at scale
- 🏆 **AgenticLabs Hackathons** — Apply your skills competitively
- 📝 **Become an Author** — Write tutorials for the community
